# 12장 실습 — 결과를 지표와 그림으로

시뮬레이션을 돌리는 것으로 일이 끝나지 않습니다. 결과를 읽고, 판단하고, 남에게 설명해야 합니다.
이 실습에서 만드는 표와 그림이 기말 프로젝트 보고서의 뼈대가 됩니다. 교재 12장에 대응합니다.

11장의 표에는 이상한 줄이 있었습니다. 차량 20대일 때 평균 대기 7.32분, 60대일 때 7.21분.
차를 세 배로 늘렸는데 대기시간이 그대로입니다. 숫자가 틀린 것이 아니라 읽는 방법이 틀린 것입니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 지표 세 무리 (교재 12.1)

지표는 누구의 편인지에 따라 셋으로 나뉩니다.

| 관점 | 무엇을 보는가 | 지표 |
|---|---|---|
| 승객 | 탈 수 있었나, 얼마나 기다렸나 | 서비스율, 대기시간 중앙값·90분위·최댓값 |
| 운영자 | 차가 일했나, 몇 명을 태웠나 | 가동률, 차량당 처리 건수 |
| 도시 | 도로를 얼마나 썼나 | 총 주행거리, 공차 비율 |

11장의 정돈본으로 80대를 돌리고 `kpi_table` 로 세 무리를 한 번에 냅니다.

In [ ]:
from smartmob.data import load_demand, load_vehicles
from smartmob.teaching.metrics import kpi_table
from smartmob.teaching.simloop import simulate

demand = load_demand("hanam")
vehicles = load_vehicles("hanam")

run = simulate(demand, vehicles, 1080, 1440)
kpi_table(run)

한 줄씩 읽습니다.

- `service_rate` 1.0 — 전원 배차받았습니다
- `wait_p50` 3.05분, `wait_p90` 8.27분 — 절반은 3분 안에 탔고, 열 명 중 아홉은 8분 안에 탔습니다
- `wait_max` 26.45분 — 가장 오래 기다린 사람입니다
- `utilization` 0.67 — 분 단위 기록에서 (운행 중 차량 ÷ 근무 차량)의 평균입니다
- `empty_share` 0.19 — 총 주행거리의 19%가 빈 차 주행입니다. 도시가 신경 쓰는 값입니다

## 2. 평균은 왜 위험한가 (교재 12.2)

차량 20, 40, 60, 80대를 돌려 `compare` 로 한 표에 놓습니다.

In [ ]:
from smartmob.teaching.metrics import compare

runs = {f"{n}대": simulate(demand, vehicles.head(n), 1080, 1440) for n in (20, 40, 60, 80)}
table = compare(runs)                 # 시나리오마다 kpi_table 을 부르고 행으로 쌓습니다
table[["service_rate", "wait_mean", "wait_p50", "wait_max", "utilization", "empty_share"]]

`wait_mean` 열만 보면 20대 7.32분, 80대 4.28분으로 차이가 크지 않습니다.
그런데 `service_rate` 를 같이 보면 20대에서는 39%만 탔습니다.

대기시간은 배차받은 사람만 계산됩니다. 못 탄 사람은 통계에서 빠집니다.
차가 적으면 가까운 승객만 배차되고 멀리 있는 승객은 포기 처리되므로, 남은 사람들의 평균 대기가 짧게 나옵니다.
이것을 생존 편향(survivorship bias)이라고 합니다.

포기한 승객을 어떻게 셀지 정해야 합니다. 두 가지 방법을 나란히 놓고 서비스율을 같이 적습니다.

In [ ]:
import numpy as np
import pandas as pd

PENALTY_MIN = 30          # 포기한 승객에게 매기는 대기시간. 30분은 정한 값입니다

rows = []
for label, result in runs.items():
    served = [r.wait_min for r in result.requests if r.pickup_time is not None]
    n_failed = sum(1 for r in result.requests if r.failed)
    rows.append({
        "시나리오": label,
        "배차된 사람만": round(np.mean(served), 2),
        f"포기={PENALTY_MIN}분으로": round(np.mean(served + [PENALTY_MIN] * n_failed), 2),
        "서비스율": round(len(served) / len(result.requests), 3),
    })
pd.DataFrame(rows)

포기한 승객에게 30분을 매기면 순서가 뒤집힙니다. 20대는 21분이 넘고 80대는 4.28분입니다.

둘 다 맞습니다. 답하려는 질문이 다를 뿐입니다.

- "택시를 잡은 사람은 얼마나 기다렸나" → 배차된 사람만
- "이 지역에서 택시를 부르면 얼마나 걸리나" → 포기까지 포함

보고서에는 서비스율과 대기시간을 항상 같이 적습니다. 하나만 적으면 읽는 사람이 속습니다.

## 3. 분포를 봅니다 (교재 12.3)

평균 대신 히스토그램을 그립니다. 20대와 80대를 겹쳐 놓습니다.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for label in ("20대", "80대"):
    waits = [r.wait_min for r in runs[label].requests if r.pickup_time is not None]
    ax.hist(waits, bins=30, alpha=0.6, label=f"{label} (배차 {len(waits)}명)")

ax.set_xlabel("대기시간 (분)"); ax.set_ylabel("승객 수")
ax.legend(); ax.grid(alpha=0.25, linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout();

80대 쪽이 왼쪽에 몰려 있고 높이도 훨씬 높습니다. 높이 차이가 곧 배차받은 사람 수의 차이입니다.
히스토그램은 평균이 숨기는 두 가지, 모양과 개수를 동시에 보여 줍니다.

## 4. 차량을 몇 대 둘 것인가 (교재 12.4)

서비스율(승객)과 가동률(운영자)을 두 축으로 놓습니다. 20대부터 80대까지 10대 간격으로 일곱 점을 찍습니다.

In [ ]:
fine = {f"{n}대": simulate(demand, vehicles.head(n), 1080, 1440)
        for n in (20, 30, 40, 50, 60, 70, 80)}
summary = compare(fine)[["service_rate", "utilization", "wait_p90", "empty_share"]]
display(summary)

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(summary["service_rate"], summary["utilization"], "-o", color="tab:blue")
for label, row in summary.iterrows():
    ax.annotate(label, (row["service_rate"], row["utilization"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)   # 점 옆에 라벨
ax.set_xlabel("서비스율 (승객이 좋아하는 것)")
ax.set_ylabel("차량 가동률 (운영자가 좋아하는 것)")
ax.grid(alpha=0.25, linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout();

오른쪽 위가 좋은데 그쪽에 점이 없습니다. 서비스율을 올리면 가동률이 떨어집니다.
어느 쪽도 다른 쪽을 완전히 이기지 못하는 선택지들을 파레토 프론트라고 합니다.

곡선이 70대에서 꺾입니다. 70대까지는 차를 더 넣을 때마다 서비스율이 오릅니다.
70대에서 80대로 가면 서비스율은 0.999에서 1.0으로 거의 그대로이고 가동률만 0.78에서 0.67로 떨어집니다.
꺾이는 지점 근처가 대개 합리적인 선택입니다.

표에서 하나 더 보입니다. 90분위 대기가 60대까지 12~14분이다가 70대에서 9.9분으로 떨어집니다.
공차 비율은 60대가 0.27로 가장 높습니다. 차가 부족할 때 멀리 있는 차를 자주 보내기 때문입니다.

## 5. 엔진 결과도 같은 표로 (교재 12.5)

우리 루프와 실제 엔진의 지표를 같은 함수로 뽑습니다. 녹화된 결과를 읽어 옵니다.

In [ ]:
from smartmob import Dtumos

engine = Dtumos().run_simulation(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)

both = compare({"내 루프": run, "DTUMOS 엔진": engine})
display(both[["service_rate", "wait_p50", "wait_p90", "utilization", "empty_share"]])
display(both[["loaded_km", "empty_km", "total_km"]])
print(f"엔진 viz_ratio: {engine.config.get('viz_ratio')}")

대기시간 분위와 공차 비율이 거의 같습니다. 서로 다른 코드가 같은 답을 낸다는 것은 좋은 신호입니다.

거리는 두 배 넘게 차이 납니다. 우리 루프가 틀린 것이 아닙니다.
엔진의 `trip.json` 은 전체 통행의 10%만 저장하도록 설정되어 있습니다(`viz_ratio=0.1`).
표본이 달라 절댓값은 비교할 수 없어도 비율은 비교할 수 있습니다. 공차 비율이 0.19 대 0.186입니다.

남의 결과와 숫자가 다르면 먼저 "같은 것을 세고 있는가"를 확인합니다.
정의가 다르거나 표본이 다른 경우가 알고리즘이 틀린 경우보다 훨씬 많습니다.

## 6. 시간대별로 보기 (교재 12.6)

한 숫자로 요약하면 시간에 따른 변화가 사라집니다. 0장에서 엔진 결과에 썼던 `plot_record` 를 우리 기록에 그대로 씁니다.

In [ ]:
from smartmob.viz import plot_record

plot_record(run.record);

record = run.record.copy()
record["hour"] = record["time"] // 60          # 자정부터의 분 → 시
by_hour = record.groupby("hour").agg(
    평균_대기승객=("waiting_passenger_cnt", "mean"),
    평균_운행차량=("driving_vehicle_cnt", "mean"),
    평균_빈차=("empty_vehicle_cnt", "mean"),
).round(1)
by_hour

대기 승객은 저녁 내내 거의 0입니다. 차 80대가 남아돌기 때문입니다.
운행 차량은 저녁 6시 44대에서 시작해 밤 11시 61대까지 늘어납니다.

## 7. 차량이 움직이는 것 보기 (교재 12.7)

숫자만으로는 어디에서 차가 부족했는지 보이지 않습니다. 엔진 결과의 통행을 지도에 놓습니다.

`trip.json` 에는 좌표가 빈 구간이 섞여 있습니다. 차량이 이미 승객 위치에 있어 이동이 없었던 경우입니다.
`prepare_trips` 가 그런 구간을 걸러 냅니다. pydeck 재생은 교재 12.7절의 `trips_deck` 으로 하고,
여기서는 오래 기다린 승객의 호출 위치를 정적인 그림으로 찍습니다.

In [ ]:
from smartmob.viz import prepare_trips

prepared = prepare_trips(engine.trips, sample=None)
print(f"전체 구간 {len(engine.trips)}개 중 그릴 수 있는 것 {len(prepared)}개")

pax = engine.passengers
long_wait = pax[pax["wait_min"] > 10]
print(f"10분 넘게 기다린 승객 {len(long_wait)}명")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(pax["origin_lon"], pax["origin_lat"], s=5, alpha=0.15, color="gray")
ax.scatter(long_wait["origin_lon"], long_wait["origin_lat"], s=30,
           color="tab:red", label="10분 초과 대기")
ax.legend(); ax.set_xticks([]); ax.set_yticks([])
ax.set_aspect(1 / 0.79)        # 위도 37.5도에서 경도 1도는 위도 1도의 0.79배 길이입니다
ax.set_title("오래 기다린 승객의 호출 위치")
fig.tight_layout();

빨간 점이 가장자리에 몰려 있습니다. 차량이 시가지 중심에 머무는 동안 외곽 호출이 밀린 것입니다.
재배치가 필요한 지점이 여기서 보입니다. 11장 연습문제 11.3이 이 문제입니다.

## 8. 결과를 웹으로 내보내기 (교재 12.8)

발표에서 보여 줄 것은 표가 아니라 움직이는 그림입니다. `export_viewer` 가 뷰어가 읽을 파일 넷을 만듭니다.
파일 이름과 키 이름은 DTUMOS 가 내는 것 그대로라, 서버에서 받은 결과 디렉터리를 그대로 넣어도 뷰어가 읽습니다.

In [ ]:
from smartmob.viz import export_viewer

# labs/ 에서 실행할 때의 경로입니다. sample=200: 브라우저가 버겁지 않게 통행 200구간만 남깁니다.
out = export_viewer(engine, "ch12_viewer/public/data", sample=200)
sorted(p.name for p in out.iterdir())

`trip.json`, `vehicle_marker.json`, `passenger_marker.json`, `meta.json` 네 파일입니다.
직접 짠 루프도 `export_viewer(run, ...)` 로 내보낼 수 있습니다. 그쪽은 도로망 위를 달리지 않으므로 통행이 직선 하나로 나옵니다.

뷰어는 `labs/ch12_viewer/` 에 있습니다. Node 20.19 이상이 필요합니다. 한 터미널에서 뷰어를 켜 둡니다.

```bash
cd labs/ch12_viewer
npm install        # 처음 한 번만
npm run dev        # 끄기 전까지 이 터미널을 차지합니다
```

브라우저가 자동으로 열립니다. 처음에는 지도만 나오고 차가 움직이지 않습니다. 빈칸이 다섯 자리 있기 때문입니다.
학생이 고치는 파일은 `src/main.js` 하나이고, 고치고 저장하면 브라우저가 알아서 다시 그립니다.

| 빈칸 | 하는 일 |
|---|---|
| `tripPath` | 구간의 좌표열을 꺼냅니다 |
| `tripTimestamps` | 각 좌표의 시각을 꺼냅니다 |
| `tripColor` | 탑승과 공차의 색을 나눕니다 |
| `waitingPassengers` | 지금 기다리는 승객만 고릅니다 |
| `nextTime` | 한 프레임만큼 시간을 흘립니다 |

네 개는 한 줄이고, 색을 정하는 `tripColor` 만 조건이 셋이라 서너 줄입니다.
`main.js` 맨 위에 채워진 예제 함수 `tripId` 가 있고, 나머지 다섯도 같은 모양입니다.
파이썬과 다른 자바스크립트 문법 여섯 가지는 교재 12.8절의 표에 있습니다.

채점은 새 터미널을 열어 저장소 뿌리에서 합니다. `npm run dev` 가 켜진 터미널에서는 안 됩니다.

```bash
node tools/check_viewer.mjs
```

Node 설치가 안 되면 12.7절의 `trips_deck` 으로 같은 통행을 노트북 안에서 재생합니다. 뷰어는 그다음에 해도 됩니다.

## 9. 빈칸

### 9.1 서비스 수준을 정합니다

"호출의 90%가 10분 안에 탄다"를 목표로 잡습니다.
4절의 `summary` 표에서 `wait_p90` 이 10분 이하가 되는 최소 차량 대수를 찾습니다.
필요하면 65대, 75대처럼 대수를 더 촘촘히 넣어 다시 돌립니다.

In [ ]:
min_fleet = None        # 90분위 대기가 10분 이하가 되는 최소 차량 대수

banner("빈칸 9.1")
todo("필요한 최소 차량 대수", min_fleet)

### 9.2 공차 주행

차량이 승객 없이 달린 거리가 전체의 몇 퍼센트인지 `empty_share` 로 봅니다.
60대와 80대를 비교하고, 60대가 더 높은 이유를 한 줄로 적습니다.
도시 전체로 보면 이 비율이 배출가스이자 혼잡입니다.

In [ ]:
empty_share_60 = None       # 60대일 때 공차 주행 비율 (0~1)
empty_share_80 = None       # 80대일 때 공차 주행 비율 (0~1)

banner("빈칸 9.2")
todo("60대 공차 비율", empty_share_60, fmt=lambda v: f"{v:.1%}")
todo("80대 공차 비율", empty_share_80, fmt=lambda v: f"{v:.1%}")

### 9.3 보고서 한 장

교재 12.9절의 여섯 항목으로 결과 절을 씁니다. 기말 프로젝트 보고서의 결론이 이 형식입니다.

1. 한 문장 결론 — "차량 70대가 하남시 저녁 수요에 적정하다"처럼
2. 근거 표 — 시나리오별 서비스율·대기 분위·가동률·공차 비율
3. 분포 그림 — 3절의 히스토그램
4. 파레토 그림 — 4절의 산점도
5. 공간 그림 — 7절의 지도
6. 한계 — 무엇을 가정했고 무엇을 넣지 않았는지

6번을 빼먹지 않습니다. 우리 시뮬레이터는 재배치가 없고, 소요시간을 직선거리로 근사했고, 신호 대기를 반영하지 않았습니다.

In [ ]:
my_conclusion = None    # 1번 결론을 한 문장으로

banner("빈칸 9.3")
todo("결론 한 문장", my_conclusion)

## 정리

- 지표는 승객·운영자·도시 세 관점으로 나눠 봅니다. 셋은 서로 부딪힙니다
- 대기시간은 배차받은 사람만 계산됩니다. 서비스율을 같이 보지 않으면 생존 편향에 속습니다
- 포기한 승객을 어떻게 셀지 정하고 그 정의를 밝힙니다. 30분을 매기면 20대의 평균 대기가 7분에서 21분이 됩니다
- 평균 대신 중앙값·90분위·분포를 봅니다
- 파레토 곡선은 70대에서 꺾입니다. 어느 점을 고를지는 데이터가 아니라 사람이 정합니다
- 남의 결과와 숫자가 다르면 알고리즘보다 정의와 표본을 먼저 확인합니다
- `export_viewer` 가 낸 파일을 `ch12_viewer/` 가 읽습니다. 채점은 새 터미널에서 `node tools/check_viewer.mjs` 입니다
- 프로젝트에서는 여러분이 고른 시군구로 이 절차를 그대로 반복합니다